In [7]:
import pandas as pd
import time

In [8]:
current_gw = 0
current_season = 20262027

In [9]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [10]:
print("Rest of Season Predictions")

predictions = db_creator.run_sql(f"""SELECT *
                                    FROM predictions
                                    WHERE season = {current_season}
                            """)

predictions.groupby('player_name_id')[['team_name','position','opp_team_elo','mins_pred',
                                           'xpoints', 'xpoints_mins', 'xpoints_goals', 
                                           'xpoints_assists', 'xpoints_clean_sheets', 
                                           'xpoints_shots_saved','xpoints_goals_conceded','xpoints_defcon','xpoints_bonus'
                                           ]].agg({'team_name':'first','position':'first','opp_team_elo':'mean','mins_pred':'mean','xpoints':'sum', 'xpoints_mins':'sum', 'xpoints_goals':'sum',
                                           'xpoints_assists':'sum', 'xpoints_clean_sheets':'sum',
                                           'xpoints_shots_saved':'sum','xpoints_goals_conceded':'sum','xpoints_defcon':'sum','xpoints_bonus':'sum'}
                                                  ).reset_index().sort_values(by='xpoints',ascending=False).head(10)

Rest of Season Predictions


,player_name_id,team_name,position,opp_team_elo,mins_pred,xpoints,xpoints_mins,xpoints_goals,xpoints_assists,xpoints_clean_sheets,xpoints_shots_saved,xpoints_goals_conceded,xpoints_defcon,xpoints_bonus
129,Declan Rice,Arsenal,MID,1066.538965,82.588767,196.799962,72.380582,55.139104,17.772681,14.221377,0.0,0.000000,19.389159,17.897059
561,William Saliba,Arsenal,DEF,1066.538965,80.976957,175.700796,71.835950,10.664544,5.773449,58.167482,0.0,-10.735909,20.037335,19.957946
58,Benjamin White,Arsenal,DEF,1066.538965,77.873167,173.760853,70.503929,10.268399,5.561671,58.167482,0.0,-10.735909,20.037335,19.957946
461,Piero Hincapié,Arsenal,DEF,1066.538965,77.203740,173.308103,70.229785,10.160080,5.491384,58.167482,0.0,-10.735909,20.037335,19.957946
188,Gabriel dos Santos Magalhães,Arsenal,DEF,1066.538965,77.203740,173.308103,70.229785,10.160080,5.491384,58.167482,0.0,-10.735909,20.037335,19.957946
106,Cristhian Mosquera,Arsenal,DEF,1066.538965,77.203740,173.308103,70.229785,10.160080,5.491384,58.167482,0.0,-10.735909,20.037335,19.957946
303,Jurriën Timber,Arsenal,DEF,1066.538965,77.203740,173.308103,70.229785,10.160080,5.491384,58.167482,0.0,-10.735909,20.037335,19.957946
383,Matheus Nunes,Man City,DEF,1067.320446,74.769482,168.858314,68.898530,11.079931,17.909514,45.558319,0.0,-15.138037,20.037335,20.512722
471,Riccardo Calafiori,Arsenal,DEF,1066.538965,71.582365,168.826400,66.904476,9.413650,5.081420,58.167482,0.0,-10.735909,20.037335,19.957946
33,Antoine Semenyo,Man City,MID,1067.320446,67.102430,168.623277,63.475400,38.861422,17.709930,11.895563,0.0,0.000000,19.389159,17.291804


In [15]:
print("Rest of Season VORP(7)")

predictions = db_creator.run_sql(f"""
WITH preds AS (
    SELECT 
        player_name_id,
        position,
        SUM(xpoints) AS xpoints
    FROM predictions
    WHERE season = {current_season}
    GROUP BY player_name_id, position
),


PlayerRankings AS (
    SELECT 
        player_name_id,
        position,
        xpoints,
        -- Get the xpoints of the player 7 spots lower in the same position
        LEAD(xpoints, 7) OVER (
            PARTITION BY position 
            ORDER BY xpoints DESC
        ) AS replacement_xpoints
    FROM 
        preds
)
SELECT 
    player_name_id,
    position,
    xpoints,
    replacement_xpoints,
    -- Calculate Value Over Replacement Player
    (xpoints - replacement_xpoints) AS vorp
FROM 
    PlayerRankings
WHERE 
    replacement_xpoints IS NOT NULL
ORDER BY 
    position, 
    vorp DESC;
    """)

predictions.sort_values(by='vorp',ascending=False).head(20)
    

Rest of Season VORP(7)


,player_name_id,position,xpoints,replacement_xpoints,vorp
300,Declan Rice,MID,196.799962,148.599644,48.200318
181,Erling Haaland,FWD,166.166534,130.534594,35.631940
182,Liam Delap,FWD,107.603631,79.800077,27.803554
183,Francisco Evanilson de Lima Barbosa,FWD,154.394203,130.484106,23.910097
301,Antoine Semenyo,MID,168.623277,148.369167,20.254111
302,Phil Foden,MID,167.538712,148.237859,19.300853
184,Ollie Watkins,FWD,146.867806,129.631726,17.236079
185,Dane Scarlett,FWD,107.752120,92.707322,15.044798
0,Tyrone Mings,DEF,157.148760,142.990383,14.158377
1,Matty Cash,DEF,156.289912,142.769850,13.520062
